# Manipulation d'image avec Pillow et NumPy

La première cellule installe les bibliothèques du TP avec pip, le gestionnaire de paquets de Python : `%pip install` installe un paquet, avec ses dépendances, dans l'environnement du notebook. Colab les préinstalle : pip répond `Requirement already satisfied` pour chacune, et n'installe rien. Sur votre machine, dans un environnement conda, l'équivalent est `conda install pillow numpy matplotlib`, dans un terminal où l'environnement est activé.

La deuxième cellule télécharge les données du TP : le jeu de données LFW (*Labeled Faces in the Wild*), 13 233 photos de visages de 250 × 250 pixels, rangées dans le dossier `lfw` avec un dossier par personne (180 Mo, que la cellule télécharge puis décompresse : comptez jusqu'à une minute), et la photo `bandana.jpg` du dernier exercice.

La troisième cellule importe les bibliothèques et définit `cote_a_cote(images, titre=None, sous_titres=None)`, qui affiche des images côte à côte : les solutions s'en servent, et vous pouvez vous en servir aussi.

In [ ]:
%pip install pillow numpy matplotlib

In [ ]:
!rm -rf sample_data
!wget -q https://ndownloader.figshare.com/files/5976018 -O lfw.tgz
!wget -q -O bandana.jpg https://github.com/shuuchuu/datasets/raw/refs/heads/main/rosie-the-riveter/rosie-the-riveter.jpg
!tar -xf lfw.tgz
!rm lfw.tgz

In [ ]:
import pathlib
from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFilter, ImageOps


def cote_a_cote(
  images: Sequence[Image.Image | np.ndarray],
  titre: str | None = None,
  sous_titres: Sequence[str] | None = None,
  dpi: int = 150,
) -> None:
  """Affiche des images côte à côte, chacune sous son sous-titre.

  Une image à un seul canal (mode `L`, ou tableau à deux dimensions) s'affiche en
  niveaux de gris, de 0 (noir) à 255 (blanc) : sans `vmin` et `vmax`, matplotlib
  étirerait les valeurs de chaque image de son minimum à son maximum, et une image
  terne paraîtrait contrastée.
  """
  fig, axes = plt.subplots(1, len(images), dpi=dpi, squeeze=False)
  for i, (ax, image) in enumerate(zip(axes[0], images)):
    tableau = np.asarray(image)
    if tableau.ndim == 2:
      ax.imshow(tableau, cmap="gray", vmin=0, vmax=255)
    else:
      ax.imshow(tableau)
    ax.axis("off")
    if sous_titres:
      ax.set_title(sous_titres[i])
  if titre:
    fig.suptitle(titre, y=0.85)
  fig.subplots_adjust(wspace=0, hspace=0)
  plt.show()

# Pillow

## Chargement d'une image

Avec la fonction `Image.open` de Pillow, chargez dans une variable `im` une image de votre choix du jeu de données, dans le dossier `lfw` (par exemple `lfw/Chris_Rock/Chris_Rock_0002.jpg`). Gardez son chemin dans une variable `chemin` : l'exercice suivant enregistre une image à côté.

Affichez la taille de l'image et son mode (l'encodage de ses couleurs), puis l'image elle-même avec la fonction `imshow` de matplotlib. Les exercices suivants partent tous de cette image `im`.

**Documentation utile**

- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `pathlib.Path("lfw/Chris_Rock/Chris_Rock_0002.jpg")` crée un chemin, que `Image.open` accepte.
- `Image.open` ne lit les pixels qu'au premier accès : dans un `with`, appelez `im.load()` avant la fin du bloc, pour vous servir de l'image après.
- `im.size` est le couple `(largeur, hauteur)` et `im.mode` le mode : toutes les photos de LFW font 250 × 250 pixels, en `RGB`.
- `plt.axis("off")` retire les graduations des axes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
chemin = pathlib.Path(...)
with Image.open(chemin) as im:
  im.load()
print(f"L'image est de dimension {...}.")
print(f"L'image est encodée en {...}.")
plt.imshow(...)
plt.axis("off")
plt.show()
```

### Solution

In [ ]:
chemin = pathlib.Path("lfw/Chris_Rock/Chris_Rock_0002.jpg")
with Image.open(chemin) as im:
  # Lit les pixels avant la fermeture du fichier : l'image reste utilisable après.
  im.load()
print(f"L'image est de dimension {im.size}.")
print(f"L'image est encodée en {im.mode}.")
plt.imshow(im)
plt.axis("off")
plt.show()

## Convertir en niveaux de gris et enregistrer

À partir de votre image `im` :

- convertissez-la en niveaux de gris (le mode `L`) avec la méthode `convert`, dans une variable `im_gris` ;
- enregistrez `im_gris` en PNG dans le **même dossier** que la photo d'origine, sous le **même nom précédé du préfixe `gris_`** (`gris_Chris_Rock_0002.png`), avec la méthode `save` ;
- affichez `im_gris` à côté de `im`, et vérifiez que le fichier est bien dans le dossier.

**Documentation utile**

- https://pillow.readthedocs.io/en/stable/handbook/concepts.html#modes
- https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.with_name
- https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.with_suffix

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `chemin.name` est le nom du fichier (`Chris_Rock_0002.jpg`), `chemin.with_name(nom)` le même chemin avec un autre nom, et `.with_suffix(".png")` change son extension.
- `save` déduit le format de l'extension `.png`.
- `cote_a_cote((im_gris, im), sous_titres=("Niveaux de gris", "Original"))` affiche les deux images côte à côte.
- Dans une cellule, `!ls {chemin_gris.parent}` liste le dossier : vous devez y voir `gris_Chris_Rock_0002.png`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
im_gris = im.convert(...)
chemin_gris = chemin.with_name(...).with_suffix(...)
im_gris.save(chemin_gris)
cote_a_cote((im_gris, im), sous_titres=("Niveaux de gris", "Original"))
```

### Solution

In [ ]:
im_gris = im.convert("L")
chemin_gris = chemin.with_name("gris_" + chemin.name).with_suffix(".png")
# Le format, PNG, se déduit de l'extension.
im_gris.save(chemin_gris)
cote_a_cote((im_gris, im), sous_titres=("Niveaux de gris", "Original"))

In [ ]:
!ls {chemin_gris.parent}

## Transformations géométriques

Effectuez et affichez les transformations suivantes de l'image `im` :

- une rotation d'un quart de tour dans le sens horaire ;
- son reflet dans un miroir (la gauche passe à droite) ;
- un agrandissement qui double sa largeur et sa hauteur, une fois avec l'interpolation `NEAREST` (le plus proche voisin) et une fois avec `BICUBIC` : comparez les contours des deux, les lettres du fond par exemple ;
- une réduction qui divise sa largeur et sa hauteur par 5.

Affichez aussi la taille des images redimensionnées.

**Documentation utile**

- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.transpose
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Transpose
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.rotate

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `im.transpose(Image.Transpose.ROTATE_270)` tourne l'image d'un quart de tour dans le sens horaire (`ROTATE_90` : dans le sens antihoraire), et `im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)` la retourne comme un miroir.
- `im.rotate(-90)` tourne aussi dans le sens horaire, mais garde la taille de départ : sur une image qui n'est pas carrée, les coins sortent du cadre et des bandes noires apparaissent, sauf avec `expand=True`.
- `im.resize((largeur, hauteur), Image.Resampling.NEAREST)` : la taille est un couple d'entiers, et `im.width` et `im.height` donnent ceux de `im`. Avec `NEAREST`, chaque pixel devient un carré de 2 × 2 pixels identiques ; `BICUBIC` calcule chaque nouveau pixel à partir de ses voisins, et adoucit les contours.
- `cote_a_cote(..., dpi=300)` affiche les images en plus grand.
- Vous devez obtenir des images de 500 × 500 et de 50 × 50 pixels.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
quart_de_tour = im.transpose(...)
miroir = im.transpose(...)
cote_a_cote(
  (quart_de_tour, miroir, im), sous_titres=("Quart de tour", "Miroir", "Original")
)

agrandie_voisin = im.resize((..., ...), Image.Resampling.NEAREST)
agrandie_bicubique = im.resize((..., ...), Image.Resampling.BICUBIC)
reduite = im.resize(...)
print(agrandie_voisin.size, reduite.size)
cote_a_cote(
  (agrandie_voisin, agrandie_bicubique), sous_titres=("NEAREST", "BICUBIC"), dpi=300
)
cote_a_cote((reduite, im), sous_titres=("Réduite", "Original"))
```

### Solution

In [ ]:
# Un quart de tour exact : rotate(-90) garderait la taille de départ, et
# rognerait une image qui n'est pas carrée.
quart_de_tour = im.transpose(Image.Transpose.ROTATE_270)
miroir = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
cote_a_cote(
  (quart_de_tour, miroir, im), sous_titres=("Quart de tour", "Miroir", "Original")
)

agrandie_voisin = im.resize((im.width * 2, im.height * 2), Image.Resampling.NEAREST)
agrandie_bicubique = im.resize((im.width * 2, im.height * 2), Image.Resampling.BICUBIC)
reduite = im.resize((im.width // 5, im.height // 5))
print(f"Agrandie : {agrandie_voisin.size}, réduite : {reduite.size}")
# NEAREST recopie chaque pixel en un carré : contours en escalier. BICUBIC les
# adoucit.
cote_a_cote(
  (agrandie_voisin, agrandie_bicubique), sous_titres=("NEAREST", "BICUBIC"), dpi=300
)
cote_a_cote((reduite, im), sous_titres=("Réduite", "Original"))

## Filtres

Un filtre de convolution remplace chaque pixel par une somme pondérée de ses voisins : ces poids forment le noyau du filtre. `ImageFilter.Kernel((3, 3), poids)` crée un filtre à partir de la taille du noyau et de ses 9 poids, ligne par ligne, et `im.filter(filtre)` l'applique.

1. Appliquez à `im` le noyau `[0, -1, 0, -1, 5, -1, 0, -1, 0]` : chaque pixel vaut cinq fois sa valeur, moins celles de ses quatre voisins directs. Affichez le résultat à côté de celui du filtre prêt à l'emploi `ImageFilter.SHARPEN` et de l'original. Que fait ce noyau ? Lequel des deux agit le plus fort ?
2. Le flou gaussien, `ImageFilter.GaussianBlur(rayon)`, remplace chaque pixel par une moyenne de ses voisins, pondérés d'autant moins qu'ils sont loin : le rayon, en pixels, règle l'étendue de ce voisinage. Appliquez-le à `im` avec un rayon de 1, de 3 et de 8, et affichez les trois résultats à côté de l'original. Lequel efface les détails du visage (les yeux, la bouche) ?

**Documentation utile**

- https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html#PIL.ImageFilter.Kernel
- https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html#PIL.ImageFilter.GaussianBlur

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Sans troisième argument, `ImageFilter.Kernel` divise la somme pondérée par la somme des poids, ici 1 : sur une zone unie, un pixel garde sa valeur ; sur un contour, son écart avec ses voisins grandit.
- `ImageFilter.SHARPEN` est un filtre prêt à l'emploi, sans parenthèses ; `ImageFilter.SHARPEN.filterargs` montre son noyau.
- `ImageFilter.GaussianBlur(3)` est une classe, qu'on instancie avec son rayon en pixels.
- `cote_a_cote(..., dpi=300)` affiche les images en plus grand, pour mieux voir les contours.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
noyau = ImageFilter.Kernel((3, 3), [...])
nette = im.filter(...)
nette_pillow = im.filter(...)
cote_a_cote(
  (nette, nette_pillow, im), sous_titres=("Noyau", "SHARPEN", "Original"), dpi=300
)

floues = [im.filter(ImageFilter.GaussianBlur(...)) for rayon in (1, 3, 8)]
cote_a_cote((*floues, im), sous_titres=("Rayon 1", "Rayon 3", "Rayon 8", "Original"))
```

### Solution

In [ ]:
# Cinq fois le pixel, moins ses quatre voisins directs : la somme des poids vaut 1,
# une zone unie garde sa valeur et un contour est accentué.
noyau = ImageFilter.Kernel((3, 3), [0, -1, 0, -1, 5, -1, 0, -1, 0])
nette = im.filter(noyau)
nette_pillow = im.filter(ImageFilter.SHARPEN)
cote_a_cote(
  (nette, nette_pillow, im), sous_titres=("Noyau", "SHARPEN", "Original"), dpi=300
)
# Les deux accentuent les contours (les cheveux, les lettres du fond). Le noyau de
# SHARPEN, 2 au centre et -1/8 sur les huit voisins, agit plus doucement que le
# nôtre.

floues = [im.filter(ImageFilter.GaussianBlur(rayon)) for rayon in (1, 3, 8)]
cote_a_cote((*floues, im), sous_titres=("Rayon 1", "Rayon 3", "Rayon 8", "Original"))
# Au rayon 1, le visage reste net ; au rayon 3, les yeux et la bouche se
# brouillent ; au rayon 8, ils sont effacés, et le visage n'est plus qu'une tache.

## Découpe et collage

- Découpez dans `im` un rectangle qui entoure le visage, avec la méthode `crop`.
- Créez une image noire de la même taille que `im` avec `Image.new`, et collez-y le visage découpé, à l'endroit où il était, avec la méthode `paste`.
- Affichez le collage à côté de l'original.

**Documentation utile**

- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.crop
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.new
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.paste

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `crop` prend une boîte `(gauche, haut, droite, bas)` en pixels, l'origine étant en haut à gauche : pour `Chris_Rock_0002.jpg`, `(75, 10, 190, 200)` entoure le visage, et donne une image de 115 × 190 pixels.
- `Image.new("RGB", im.size)` crée une image noire de la taille de `im`.
- `collage.paste(visage, (gauche, haut))` colle `visage` avec son coin en haut à gauche en `(gauche, haut)` : elle modifie `collage` sur place, et renvoie `None`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
cadre_visage = (..., ..., ..., ...)  # (gauche, haut, droite, bas)
visage = im.crop(cadre_visage)
collage = Image.new(...)
collage.paste(visage, ...)
cote_a_cote((collage, im), "Découpe et collage", ("Collage", "Original"))
```

### Solution

In [ ]:
cadre_visage = (75, 10, 190, 200)  # (gauche, haut, droite, bas)
visage = im.crop(cadre_visage)

# Une image noire de la taille de l'originale, où le visage revient à sa place :
# son coin en haut à gauche en (gauche, haut).
collage = Image.new("RGB", im.size)
collage.paste(visage, cadre_visage[:2])

cote_a_cote((collage, im), "Découpe et collage", ("Collage", "Original"))

## Isolation des canaux de couleurs

La méthode `split` de Pillow sépare les canaux d'une image : chacun devient une `Image` en mode `L` (niveaux de gris).

- Séparez les canaux rouge, vert et bleu de `im`.
- Affichez chacun d'eux en niveaux de gris. Voyez-vous des différences ?
- Refusionnez les canaux avec `Image.merge`, en échangeant le rouge et le bleu, et affichez le résultat.
- Affichez de nouveau chaque canal, le rouge en rouge plutôt qu'en gris, et de même pour le vert et le bleu.

**Documentation utile**

- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.merge

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `r, g, b = im.split()` ; affiché en niveaux de gris, un canal est clair là où sa couleur est forte.
- `Image.merge("RGB", (b, g, r))` reconstruit une image à partir de trois canaux, dans l'ordre donné.
- Pour voir le canal rouge en rouge, fusionnez-le avec deux canaux noirs : `noir = Image.new("L", im.size)`, puis `Image.merge("RGB", (r, noir, noir))`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
r, g, b = im.split()
cote_a_cote((r, g, b), sous_titres=("rouge", "vert", "bleu"))

canaux_inverses = Image.merge("RGB", ...)
cote_a_cote((canaux_inverses, im), sous_titres=("Rouge et bleu échangés", "Original"))

noir = Image.new("L", im.size)
rouge = Image.merge("RGB", (r, noir, noir))
vert = ...
bleu = ...
cote_a_cote((rouge, vert, bleu), sous_titres=("rouge", "vert", "bleu"), dpi=300)
```

### Solution

In [ ]:
r, g, b = im.split()
# Le tapis rouge, en bas à gauche, est clair dans le canal rouge et sombre dans
# les deux autres.
cote_a_cote((r, g, b), sous_titres=("rouge", "vert", "bleu"))

In [ ]:
canaux_inverses = Image.merge("RGB", (b, g, r))
cote_a_cote((canaux_inverses, im), sous_titres=("Rouge et bleu échangés", "Original"))

In [ ]:
noir = Image.new("L", im.size)
rouge = Image.merge("RGB", (r, noir, noir))
vert = Image.merge("RGB", (noir, g, noir))
bleu = Image.merge("RGB", (noir, noir, b))
cote_a_cote((rouge, vert, bleu), sous_titres=("rouge", "vert", "bleu"), dpi=300)

# Manipulation d'image avec NumPy (*prérequis : NumPy et matplotlib*)

Si Pillow propose de nombreuses méthodes de manipulation d'images, la bibliothèque s'en tient surtout aux opérations classiques et aux fonctions de chargement, de sauvegarde et de changement de mode. Elle manque d'opérations avancées et de souplesse.

NumPy, bibliothèque de calcul scientifique, permet d'écrire des opérations plus complexes en traitant les images comme des tableaux multidimensionnels.

## Refaire les transformations de Pillow avec NumPy

Une image Pillow se convertit en tableau NumPy de forme `(hauteur, largeur, canaux)` avec `np.asarray` : la cellule de l'exercice crée ce tableau, `im_np`, et affiche sa forme et son type. Les transformations de la partie Pillow deviennent alors des opérations sur ce tableau : de l'arithmétique et des tranches.

Refaites avec NumPy, à partir de `im_np` seul, quatre transformations que vous avez faites avec Pillow, et comparez chacune au résultat de Pillow :

- les niveaux de gris : Pillow calcule chaque pixel gris par la somme pondérée `0.299 * rouge + 0.587 * vert + 0.114 * bleu` de ses trois canaux, un produit matriciel ; comparez votre image à `im_gris` par l'écart maximal entre les deux ;
- le négatif, où chaque valeur $v$ devient $255 - v$ ; comparez-le à `ImageOps.invert(im)` ;
- le miroir ; comparez-le à `miroir` ;
- le recadrage sur le visage, par tranches, avec le cadre `cadre_visage` de l'exercice de découpe ; comparez-le à `visage`.

`np.array_equal` dit si deux tableaux sont identiques. Affichez aussi vos quatre images.

Critère de réussite : l'écart maximal des niveaux de gris est d'un demi-niveau environ (0,495 pour `Chris_Rock_0002.jpg`), l'arrondi de Pillow à l'entier, et les trois autres comparaisons valent `True`.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
- https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html
- https://pillow.readthedocs.io/en/stable/reference/ImageOps.html#PIL.ImageOps.invert

In [ ]:
im_np = np.asarray(im)
print(f"Forme : {im_np.shape}, type : {im_np.dtype}")

# Votre code ici

### Conseils supplémentaires

- `im_np @ [0.299, 0.587, 0.114]` multiplie le dernier axe de `im_np`, celui des trois canaux, par les trois poids : un tableau de forme `(250, 250)`, de nombres à virgule.
- `np.abs(gris_np - np.asarray(im_gris)).max()` donne l'écart maximal entre deux tableaux.
- `255 - im_np` reste en `uint8` : chaque résultat est entre 0 et 255, rien ne déborde.
- `im_np[:, ::-1]` parcourt les colonnes à l'envers.
- `im_np[haut:bas, gauche:droite]` recadre, les lignes d'abord, alors que le cadre de Pillow, `(gauche, haut, droite, bas)`, donne les colonnes d'abord.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
gris_np = im_np @ [...]
ecart = np.abs(gris_np - np.asarray(im_gris)).max()
print(f"Écart maximal des niveaux de gris : {ecart:.3f}")

negatif_np = ...
print(
  "Négatif identique :", np.array_equal(negatif_np, np.asarray(ImageOps.invert(im)))
)

miroir_np = ...
print("Miroir identique :", np.array_equal(miroir_np, np.asarray(miroir)))

gauche, haut, droite, bas = cadre_visage
visage_np = ...
print("Visage identique :", np.array_equal(visage_np, np.asarray(visage)))

cote_a_cote(
  (gris_np, negatif_np, miroir_np, visage_np),
  sous_titres=("Niveaux de gris", "Négatif", "Miroir", "Visage"),
)
```

### Solution

In [ ]:
# Le produit matriciel pondère les trois canaux (le dernier axe) de chaque pixel.
gris_np = im_np @ [0.299, 0.587, 0.114]
# Pillow arrondit chaque pixel à l'entier : un demi-niveau d'écart, environ.
ecart = np.abs(gris_np - np.asarray(im_gris)).max()
print(f"Écart maximal des niveaux de gris : {ecart:.3f}")

negatif_np = 255 - im_np
print(
  "Négatif identique :", np.array_equal(negatif_np, np.asarray(ImageOps.invert(im)))
)

# Les colonnes à l'envers.
miroir_np = im_np[:, ::-1]
print("Miroir identique :", np.array_equal(miroir_np, np.asarray(miroir)))

# Les lignes d'abord (de haut à bas), puis les colonnes (de gauche à droite).
gauche, haut, droite, bas = cadre_visage
visage_np = im_np[haut:bas, gauche:droite]
print("Visage identique :", np.array_equal(visage_np, np.asarray(visage)))

cote_a_cote(
  (gris_np, negatif_np, miroir_np, visage_np),
  sous_titres=("Niveaux de gris", "Négatif", "Miroir", "Visage"),
)

## Échanger et isoler les canaux avec NumPy

Le dernier axe de `im_np` est celui des canaux, dans l'ordre rouge, vert, bleu. Reconstruisez avec NumPy, à partir de `im_np` seul, deux images de l'exercice d'isolation des canaux avec Pillow :

- `canaux_inverses`, où le rouge et le bleu sont échangés : une seule tranche, sur le dernier axe, suffit ;
- `rouge`, le canal rouge seul, les deux autres à 0 : partez d'un tableau de zéros de la forme et du type de `im_np`, créé par `np.zeros_like`, et copiez-y le canal rouge par une affectation.

Vérifiez avec `np.array_equal` que chacune est identique à l'image de Pillow, et affichez-les.

Critère de réussite : les deux comparaisons valent `True`.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.zeros_like.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `im_np[:, :, k]` garde toutes les lignes, toutes les colonnes et le seul canal `k` (0 pour le rouge, 1 pour le vert, 2 pour le bleu).
- `im_np[:, :, ::-1]` parcourt les canaux à l'envers : bleu, vert, rouge.
- `np.zeros_like(im_np)` crée un tableau de zéros de même forme et de même type (`uint8`), que l'on peut modifier, contrairement à `im_np` : un tableau créé par `np.asarray` à partir d'une image Pillow est en lecture seule, et une affectation y lève `ValueError: assignment destination is read-only`.
- `rouge_np[:, :, 0] = im_np[:, :, 0]` copie le canal rouge dans le premier canal de `rouge_np`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
canaux_inverses_np = im_np[...]
identiques = np.array_equal(canaux_inverses_np, np.asarray(canaux_inverses))
print("Canaux échangés identiques :", identiques)

rouge_np = np.zeros_like(...)
rouge_np[...] = ...
print("Canal rouge identique :", np.array_equal(rouge_np, np.asarray(rouge)))

cote_a_cote(
  (canaux_inverses_np, rouge_np), sous_titres=("Rouge et bleu échangés", "Rouge seul")
)
```

### Solution

In [ ]:
# Les canaux à l'envers : bleu, vert, rouge.
canaux_inverses_np = im_np[:, :, ::-1]
identiques = np.array_equal(canaux_inverses_np, np.asarray(canaux_inverses))
print("Canaux échangés identiques :", identiques)

# im_np, créé par np.asarray, est en lecture seule : on part d'un tableau de zéros,
# que l'on peut modifier, et l'on y copie le canal rouge.
rouge_np = np.zeros_like(im_np)
rouge_np[:, :, 0] = im_np[:, :, 0]
print("Canal rouge identique :", np.array_equal(rouge_np, np.asarray(rouge)))

cote_a_cote(
  (canaux_inverses_np, rouge_np), sous_titres=("Rouge et bleu échangés", "Rouge seul")
)

## Histogramme d'une image

Un outil classique du traitement d'image est l'[histogramme des valeurs de pixels](https://fr.wikipedia.org/wiki/Histogramme_(imagerie_num%C3%A9rique)).

1. Convertissez votre image en niveaux de gris, et gardez le tableau NumPy associé dans `im_gris_np`.
2. Écrivez la fonction `tracer_histogramme(image)`, qui trace l'histogramme d'une image en niveaux de gris :
   - en abscisse, les valeurs possibles d'un pixel (0 à 255) ;
   - en ordonnée, le nombre de pixels de chaque valeur ;
   - sur le même graphique, la courbe de la somme cumulée de l'histogramme : en $k$, le nombre de pixels de valeur inférieure ou égale à $k$.

   Pour l'histogramme, deux manières : compter les valeurs vous-même avec `np.histogram` et tracer les barres avec `plt.bar`, ou laisser `plt.hist` compter.
3. Tracez l'histogramme de `im_gris_np`.

L'exercice suivant se sert de `tracer_histogramme`.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.histogram.html
- https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

In [ ]:
def tracer_histogramme(image: np.ndarray) -> None:
  # Votre code ici
  ...

### Conseils supplémentaires

- `np.array(im.convert("L"))` donne le tableau en niveaux de gris, de forme `(250, 250)`.
- `effectifs, bornes = np.histogram(image, bins=256, range=(0, 256))` compte les pixels de chaque valeur : avec `range=(0, 256)`, les 256 classes ont une largeur de 1, et `effectifs[k]` est le nombre de pixels de valeur `k`. Sans `range`, les classes ne couvrent que l'intervalle entre le minimum et le maximum de l'image, et `effectifs[k]` ne correspond plus à la valeur `k`.
- `plt.hist(image.ravel(), bins=256, range=(0, 256))` compte et trace d'un coup ; `ravel()` aplatit le tableau.
- `cumul = np.cumsum(effectifs)` donne la somme cumulée. Son dernier élément est le nombre de pixels, bien plus grand que les barres : multipliez-la par `effectifs.max() / cumul[-1]` pour la tracer à la même échelle, avec `plt.plot(np.arange(256), ...)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def tracer_histogramme(image: np.ndarray) -> None:
  effectifs, _ = np.histogram(image, bins=256, range=(0, 256))
  cumul = ...
  plt.hist(..., bins=256, range=(0, 256))
  plt.plot(np.arange(256), ...)
  plt.show()


im_gris_np = np.array(...)
tracer_histogramme(im_gris_np)
```

### Solution

In [ ]:
def tracer_histogramme(image: np.ndarray) -> None:
  # range=(0, 256) : effectifs[k] est le nombre de pixels de valeur k.
  effectifs, _ = np.histogram(image, bins=256, range=(0, 256))
  cumul = np.cumsum(effectifs)
  plt.hist(image.ravel(), bins=256, range=(0, 256))
  # La somme cumulée, ramenée à la hauteur de la plus grande barre pour ne pas
  # écraser l'histogramme.
  plt.plot(np.arange(256), cumul * effectifs.max() / cumul[-1])
  plt.xlabel("Valeur du pixel")
  plt.ylabel("Nombre de pixels")
  plt.show()


im_gris_np = np.array(im.convert("L"))
tracer_histogramme(im_gris_np)

### Histogramme d'une image en couleur

Une image en couleur a un histogramme par canal, ici superposés ; le TP scikit-image les fait calculer par `exposure.histogram`.

In [ ]:
for canal, couleur, nom in [(0, "r", "rouge"), (1, "g", "vert"), (2, "b", "bleu")]:
  plt.hist(
    im_np[:, :, canal].ravel(),
    bins=256,
    range=(0, 256),
    color=couleur,
    alpha=0.5,
    label=nom,
  )
plt.legend()
plt.show()

## Corriger le contraste par égalisation d'histogramme

Une utilisation simple de l'histogramme est la correction du contraste. Selon l'image, l'histogramme peut se concentrer sur les noirs, les gris ou les blancs, sans occuper toute la plage des valeurs possibles. Pour l'essayer sur une image vraiment terne, la cellule de l'exercice crée `terne` : votre image en niveaux de gris, ramenée aux valeurs de 80 à 165 au plus.

L'égalisation remplace chaque valeur de pixel $k$ par :

$$
T(k) = \frac{255}{n}\sum_{j=0}^k n_j
$$

avec :

- $k$ la valeur d'un pixel (entre 0 et 255) ;
- $n_j$ le nombre de pixels de valeur $j$ ;
- $n$ le nombre de pixels ;
- $\sum_{j=0}^k n_j$ la fréquence cumulée des valeurs de pixels de 0 à $k$.

$T(k)$ ne dépend que de $k$ : plutôt que de la calculer pixel par pixel, calculez-la une fois pour chacune des 256 valeurs possibles, dans un tableau `table` (`table[k]` vaut $T(k)$), puis appliquez-la à toute l'image d'un coup par indexation avancée : `table[image]` remplace chaque pixel de valeur `k` par `table[k]`.

1. Écrivez la fonction `egalisation(image)`, qui renvoie l'image égalisée.
2. Appliquez-la à `terne`, et affichez le résultat à côté de `terne`.
3. Tracez l'histogramme de `terne`, puis celui du résultat, avec `tracer_histogramme`.

Critère de réussite : les valeurs de l'image égalisée vont de presque 0 à 255 (de 1 à 255 pour `Chris_Rock_0002.jpg` : affichez son minimum et son maximum), et sa courbe cumulée, qui montait d'un coup pour `terne`, est presque une droite.

**Documentation utile**

- https://numpy.org/doc/stable/user/basics.indexing.html#integer-array-indexing

In [ ]:
# Votre image en niveaux de gris, ramenée aux valeurs de 80 à 165 au plus.
terne = 80 + im_gris_np // 3
print(f"Valeurs de terne : de {terne.min()} à {terne.max()}")


def egalisation(image: np.ndarray) -> np.ndarray:
  # Votre code ici
  ...

### Conseils supplémentaires

- Les effectifs $n_j$ : `np.histogram(image, bins=256, range=(0, 256))`, avec `range` (voir les conseils de l'exercice précédent).
- `cumul = np.cumsum(effectifs)` : `cumul[k]` vaut $\sum_{j=0}^k n_j$, et `cumul[-1]` vaut $n$.
- `(255 * cumul / cumul[-1]).astype(np.uint8)` calcule les 256 valeurs de $T$ d'un coup.
- `table[image]` a la forme de `image` : chaque entier de `image` sert d'indice dans `table`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def egalisation(image: np.ndarray) -> np.ndarray:
  effectifs, _ = np.histogram(...)
  cumul = ...
  table = ...
  return table[image]


egalisee = egalisation(terne)
print(egalisee.min(), egalisee.max())
cote_a_cote((terne, egalisee), sous_titres=("Terne", "Égalisée"))
tracer_histogramme(terne)
tracer_histogramme(egalisee)
```

### Solution

In [ ]:
def egalisation(image: np.ndarray) -> np.ndarray:
  """Renvoie l'image en niveaux de gris `image`, son histogramme égalisé."""
  effectifs, _ = np.histogram(image, bins=256, range=(0, 256))
  cumul = np.cumsum(effectifs)
  # table[k] vaut T(k), pour chacune des 256 valeurs possibles d'un pixel.
  table = (255 * cumul / cumul[-1]).astype(np.uint8)
  # Indexation avancée : chaque pixel de valeur k devient table[k].
  return table[image]


egalisee = egalisation(terne)
print(f"Valeurs de l'image égalisée : de {egalisee.min()} à {egalisee.max()}")
cote_a_cote(
  (terne, egalisee, im_gris_np), sous_titres=("Terne", "Égalisée", "Niveaux de gris")
)
tracer_histogramme(terne)
tracer_histogramme(egalisee)

## Fusion d'images (difficulté +)

Fusionnez deux images l'une dans l'autre, progressivement de gauche à droite : à gauche, on ne voit que la première ; à droite, que la seconde ; au milieu, les deux à 50 %. La colonne $j$ du résultat mélange les colonnes $j$ des deux images, $a_j$ et $b_j$ :

$$
\alpha_j \cdot a_j + (1 - \alpha_j) \cdot b_j
$$

où $\alpha_j$ décroît régulièrement de 1, pour la première colonne, à 0, pour la dernière.

1. Chargez deux images de votre choix dans des tableaux NumPy `im_1` et `im_2`. Elles doivent avoir la même taille : toutes les photos de LFW font 250 × 250 pixels.
2. Écrivez la fonction `fusion(im_1, im_2)`, sans boucle : rangez les $\alpha_j$ dans un tableau, et laissez la diffusion (*broadcasting*) de NumPy appliquer chacun à toute sa colonne, sur toutes les lignes et tous les canaux.
3. Affichez le résultat entre les deux images.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.linspace.html
- https://numpy.org/doc/stable/user/basics.broadcasting.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `np.linspace(1, 0, largeur)` renvoie `largeur` valeurs régulièrement espacées, de 1 à 0 : les $\alpha_j$, dans un tableau de forme `(largeur,)`.
- Une image est de forme `(hauteur, largeur, 3)` : `alpha[None, :, None]`, de forme `(1, largeur, 1)`, se diffuse sur toutes les lignes et tous les canaux, et `alpha * im_1` multiplie chaque colonne $j$ par $\alpha_j$.
- Le résultat est un tableau de nombres à virgule : `.astype(np.uint8)` le ramène à des entiers de 0 à 255.
- Si la diffusion vous résiste, écrivez d'abord une boucle sur les colonnes, puis remplacez-la : `resultat = np.zeros(im_1.shape)`, puis pour chaque `j`, `resultat[:, j] = alpha[j] * im_1[:, j] + (1 - alpha[j]) * im_2[:, j]`.
- Vérification : la première colonne du résultat est celle de `im_1`, et la dernière celle de `im_2` (`np.array_equal(resultat[:, 0], im_1[:, 0])`).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def fusion(im_1: np.ndarray, im_2: np.ndarray) -> np.ndarray:
  largeur = im_1.shape[1]
  alpha = np.linspace(...)[None, :, None]
  return (...).astype(np.uint8)


im_1 = np.array(Image.open(...))
im_2 = np.array(Image.open(...))
cote_a_cote((im_1, fusion(im_1, im_2), im_2), sous_titres=("im_1", "Fusion", "im_2"))
```

### Solution

In [ ]:
def fusion(im_1: np.ndarray, im_2: np.ndarray) -> np.ndarray:
  largeur = im_1.shape[1]
  # alpha passe de 1, à gauche, à 0, à droite ; sa forme (1, largeur, 1) le diffuse
  # sur toutes les lignes et tous les canaux.
  alpha = np.linspace(1, 0, largeur)[None, :, None]
  return (alpha * im_1 + (1 - alpha) * im_2).astype(np.uint8)


im_1 = np.array(Image.open("lfw/Chris_Rock/Chris_Rock_0001.jpg"))
im_2 = np.array(Image.open("lfw/Chris_Rock/Chris_Rock_0002.jpg"))
cote_a_cote((im_1, fusion(im_1, im_2), im_2), sous_titres=("im_1", "Fusion", "im_2"))

## Isolation de couleurs (difficulté ++)

Mettons à présent toutes vos compétences en pratique, avec Pillow et NumPy.

Vous trouvez à la racine de votre dossier un fichier `bandana.jpg`, l'image ci-dessous :
![bandana](https://github.com/shuuchuu/datasets/raw/refs/heads/main/rosie-the-riveter/rosie-the-riveter.jpg)

*Photo : Alfred T. Palmer, 1943, domaine public.*

L'objectif de cet exercice est de produire une image en niveaux de gris qui garde le rouge du bandana.

Voici les informations dont vous aurez besoin :

- Une image `RGB` dont les trois canaux sont identiques est une image en niveaux de gris.
- Le mode `RGB` se prête mal au choix d'une couleur : préférez-lui le mode [`HSV`](https://fr.wikipedia.org/wiki/Teinte_saturation_lumi%C3%A8re). Son premier canal, `H` (la teinte), donne la couleur du pixel sur un cercle de couleurs. Pillow l'exprime de 0 à 255 : 0 correspond au rouge, et 255 aussi (quoique très subtilement plus violacé). Le cyan est donc autour de 128, à l'opposé (ne vous fiez pas aux valeurs indiquées sur le dessin ci-dessous).
  ![](https://upload.wikimedia.org/wikipedia/commons/5/5c/Color_wheel_with_degree.png)
- Le deuxième canal, `S` (la saturation), sépare une couleur vive d'une couleur terne de même teinte : sans lui, la peau, d'une teinte proche du rouge, garde un peu de sa couleur.
- Une image Pillow se transforme en tableau NumPy avec `np.array`, et un tableau en image Pillow avec `Image.fromarray`.

Critère de réussite : le bandana garde son rouge en entier, et le reste de l'image passe en niveaux de gris, à part les ongles et les lèvres, rouges eux aussi.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.where.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `np.array(Image.fromarray(bandana).convert("HSV"))` donne un tableau de forme `(hauteur, largeur, 3)` : `[:, :, 0]` est la teinte, `[:, :, 1]` la saturation.
- Le cercle des teintes passe par 0 : le rouge est à la fois près de 255 et près de 0. Un seul intervalle, `(teinte > 235) & (teinte < 251)`, en oublie la moitié, et le bandana ressort en taches grises : il faut l'union de deux intervalles, `(teinte > 235) | (teinte < 12)`.
- `& (saturation > 90)` écarte les tons peu saturés.
- Le masque obtenu est un tableau de booléens de forme `(hauteur, largeur)` : `masque[:, :, None]` lui ajoute un axe de canal, pour qu'il se diffuse sur les trois canaux de l'image, et de même pour l'image en niveaux de gris.
- `np.where(masque, image, gris)` prend `image` là où `masque` est vrai, et `gris` ailleurs.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bandana = np.array(Image.open("bandana.jpg"))
hsv = np.array(Image.fromarray(bandana).convert(...))
teinte, saturation = hsv[:, :, 0], hsv[:, :, 1]
masque_rouge = (...) & (saturation > 90)
gris = np.array(Image.fromarray(bandana).convert(...))
resultat = np.where(masque_rouge[:, :, None], bandana, gris[:, :, None])
cote_a_cote((bandana, resultat))
```

### Solution

In [ ]:
def isoler_couleur(image: np.ndarray, masque: np.ndarray) -> np.ndarray:
  """Garde la couleur des pixels du masque, et passe les autres en niveaux de gris."""
  gris = np.array(Image.fromarray(image).convert("L"))
  # Un axe de canal, de longueur 1, que NumPy diffuse sur les trois canaux de
  # l'image : (hauteur, largeur) devient (hauteur, largeur, 1).
  return np.where(masque[:, :, None], image, gris[:, :, None])


bandana = np.array(Image.open("bandana.jpg"))
# HSV, pour choisir une couleur par sa teinte.
hsv = np.array(Image.fromarray(bandana).convert("HSV"))
teinte, saturation = hsv[:, :, 0], hsv[:, :, 1]
# Le rouge est aux deux bouts du cercle des teintes, près de 255 et près de 0 ; la
# saturation écarte la peau, d'une teinte proche mais plus terne.
masque_rouge = ((teinte > 235) | (teinte < 12)) & (saturation > 90)
cote_a_cote((bandana, isoler_couleur(bandana, masque_rouge)))